# Prepare Data for Qwen2.5-7B Finetuning

This notebook converts the synthesized ToM questions into the format needed for finetuning.

In [1]:
import pandas as pd
import json
import glob
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


## Step 1: Load and Concatenate Synthesized Data

In [2]:
# Load all CSV files from synthesized data directory
synthesized_data_dir = "../results/synthesized_data/data"
csv_files = glob.glob(f"{synthesized_data_dir}/*.csv")

print(f"Found {len(csv_files)} CSV files in {synthesized_data_dir}:")
for f in csv_files:
    print(f"  - {Path(f).name}")

# Load and concatenate all CSV files
synthesized_dfs = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    synthesized_dfs.append(df)
    print(f"Loaded {Path(csv_file).name}: {len(df)} rows")

if synthesized_dfs:
    synthesized_df = pd.concat(synthesized_dfs, ignore_index=True)
    print(f"\n✓ Concatenated {len(synthesized_dfs)} files: {len(synthesized_df)} total rows")
    print(f"Columns: {list(synthesized_df.columns)}")
else:
    raise ValueError(f"No CSV files found in {synthesized_data_dir}")

Found 12 CSV files in ../results/synthesized_data/data:
  - cluster45_30questions.csv
  - cluster4_80questions.csv
  - cluster5_10Q.csv
  - cluster5_11Qs.csv
  - cluster5_20Q.csv
  - cluster5_30Q.csv
  - cluster9_10q.csv
  - r2_cluster4_100Q.csv
  - r2_cluster4_200Q.csv
  - r2_cluster5_80Q.csv
  - r2_cluster5_120Q.csv
  - r2_cluster9_100Q.csv
Loaded cluster45_30questions.csv: 30 rows
Loaded cluster4_80questions.csv: 80 rows
Loaded cluster5_10Q.csv: 10 rows
Loaded cluster5_11Qs.csv: 11 rows
Loaded cluster5_20Q.csv: 20 rows
Loaded cluster5_30Q.csv: 30 rows
Loaded cluster9_10q.csv: 10 rows
Loaded r2_cluster4_100Q.csv: 100 rows
Loaded r2_cluster4_200Q.csv: 200 rows
Loaded r2_cluster5_80Q.csv: 80 rows
Loaded r2_cluster5_120Q.csv: 120 rows
Loaded r2_cluster9_100Q.csv: 100 rows

✓ Concatenated 12 files: 791 total rows
Columns: ['cluster_id', 'question_index', 'story', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'correct_answer', 'cluster_summary']


## Step 2: Add TASK Column and Standardize Column Names

In [3]:
# Add TASK column based on cluster_id
# cluster_id 3 = table cluster 4, cluster_id 4 = table cluster 5
synthesized_df['TASK'] = synthesized_df['cluster_id'].apply(
    lambda x: f"synthesize data for cluster {x+1}"
)

print(f"✓ Added TASK column")
print(f"Task distribution:")
print(synthesized_df['TASK'].value_counts())

# Standardize column names to match train.csv format (uppercase with hyphens)
column_mapping = {
    'story': 'STORY',
    'question': 'QUESTION',
    'option_a': 'OPTION-A',
    'option_b': 'OPTION-B',
    'option_c': 'OPTION-C',
    'option_d': 'OPTION-D',
    'correct_answer': 'ANSWER'
}

synthesized_df = synthesized_df.rename(columns=column_mapping)

print(f"\n✓ Standardized column names")
print(f"Columns after standardization: {list(synthesized_df.columns)}")

✓ Added TASK column
Task distribution:
TASK
synthesize data for cluster 4    400
synthesize data for cluster 5    281
synthesize data for cluster 9    110
Name: count, dtype: int64

✓ Standardized column names
Columns after standardization: ['cluster_id', 'question_index', 'STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'cluster_summary', 'TASK']


## Step 3: Load Original Train Data

In [4]:
# Load train.csv
train_csv_path = "../train.csv"
train_df = pd.read_csv(train_csv_path)

## Step 4: Concatenate Datasets (Shared Columns Only)

In [5]:
# Find shared columns (case-insensitive comparison)
train_cols_lower = {col.lower(): col for col in train_df.columns}
synth_cols_lower = {col.lower(): col for col in synthesized_df.columns}

# Get shared column names (using train.csv naming convention)
shared_cols = []
for col_lower in train_cols_lower.keys():
    if col_lower in synth_cols_lower:
        train_col = train_cols_lower[col_lower]
        synth_col = synth_cols_lower[col_lower]
        shared_cols.append(train_col)
        
        # Rename synthesized column to match train.csv if different
        if synth_col != train_col:
            synthesized_df = synthesized_df.rename(columns={synth_col: train_col})

print(f"✓ Found {len(shared_cols)} shared columns:")
print(f"  {shared_cols}")

# Select only shared columns from both dataframes
train_df_subset = train_df[shared_cols]
synthesized_df_subset = synthesized_df[shared_cols]

# Concatenate
combined_df = pd.concat([train_df_subset, synthesized_df_subset], ignore_index=True)

print(f"\n✓ Combined dataset:")
print(f"  Train data: {len(train_df_subset)} rows")
print(f"  Synthesized data: {len(synthesized_df_subset)} rows")
print(f"  Total: {len(combined_df)} rows")
print(f"  Columns: {list(combined_df.columns)}")

✓ Found 8 shared columns:
  ['STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'TASK']

✓ Combined dataset:
  Train data: 2003 rows
  Synthesized data: 791 rows
  Total: 2794 rows
  Columns: ['STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'TASK']


In [6]:
# Modify ANSWER column: wrap each answer in double brackets
# Example: "B" -> "[[B]]"
combined_df['ANSWER'] = combined_df['ANSWER'].apply(lambda x: f"[[{x}]]")
combined_df = combined_df.sample(frac=1).reset_index(drop=True)

In [7]:
combined_df.to_csv("finetune_data.csv", index=False)

## Step 5

In [8]:
def create_training_sample(row):
    """
    Convert a single row to Qwen2.5 training format.
    Matches the format from ToM/prompt-noCoT.txt
    """
    # Format the question following prompt-noCoT.txt structure
    user_message = f"""Below is a multiple-choice question with a story and serveral answer options. Based on the content of the story and the given question, please infer the most likely answer and output the answer index.

Note:
(1) Please only output the most likely answer index in the format: [[Answer Index]], for example, if the most likely answer option is 'A. Handbag', then output '[[A]]';
(2) You must choose one of the given answer options 'A, B, C, D' as the most likely answer, regardless of whether the story provides enough information. If you think there is not enough information in the story to choose an answer, please output the most likely answer among "[[A]]", "[[B]]", "[[C]]", or "[[D]]" based on the current story;
(3) Please only output the most likely answer index based on the given information, and do not output any other content.

[Story]
{row['STORY']}

[Question]
{row['QUESTION']}

[Candidate Answers]
A. {row['OPTION-A']} B. {row['OPTION-B']} C. {row['OPTION-C']} D. {row['OPTION-D']}

Output the most likely answer among "[[A]]", "[[B]]", "[[C]]", or "[[D]]", and nothing else. Do not think."""
    
    # The correct answer (already wrapped in [[]] from previous step)
    assistant_message = row['ANSWER']
    
    return {
        "messages": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
    }

# Convert all rows
training_data = [create_training_sample(row) for _, row in combined_df.iterrows()]

print(f"✓ Created {len(training_data)} training samples")
print(f"\nExample sample:")
print(json.dumps(training_data[0], indent=2, ensure_ascii=False))

✓ Created 2794 training samples

Example sample:
{
  "messages": [
    {
      "role": "user",
      "content": "Below is a multiple-choice question with a story and serveral answer options. Based on the content of the story and the given question, please infer the most likely answer and output the answer index.\n\nNote:\n(1) Please only output the most likely answer index in the format: [[Answer Index]], for example, if the most likely answer option is 'A. Handbag', then output '[[A]]';\n(2) You must choose one of the given answer options 'A, B, C, D' as the most likely answer, regardless of whether the story provides enough information. If you think there is not enough information in the story to choose an answer, please output the most likely answer among \"[[A]]\", \"[[B]]\", \"[[C]]\", or \"[[D]]\" based on the current story;\n(3) Please only output the most likely answer index based on the given information, and do not output any other content.\n\n[Story]\nZhou Yu is a lawyer who

## Step 6: Split into Train/Validation

In [9]:
from sklearn.model_selection import train_test_split

# Split 90% train, 10% validation
train_data, val_data = train_test_split(training_data, test_size=0.1, random_state=42)

print(f"✓ Split complete:")
print(f"  Train samples: {len(train_data)}")
print(f"  Validation samples: {len(val_data)}")

✓ Split complete:
  Train samples: 2514
  Validation samples: 280


In [10]:
output_dir = Path("../finetune")
output_dir.mkdir(exist_ok=True)

# Save train data
train_path = output_dir / "train_r2.jsonl"
with open(train_path, 'w', encoding='utf-8') as f:
    for sample in train_data:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')

# Save validation data
val_path = output_dir / "val_r2.jsonl"
with open(val_path, 'w', encoding='utf-8') as f:
    for sample in val_data:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')

print(f"✓ Saved training data to {train_path}")
print(f"✓ Saved validation data to {val_path}")

✓ Saved training data to ../finetune/train_r2.jsonl
✓ Saved validation data to ../finetune/val_r2.jsonl


# Finetune without synthesized data

In [5]:
# Find shared columns (case-insensitive comparison)
train_cols_lower = {col.lower(): col for col in train_df.columns}
synth_cols_lower = {col.lower(): col for col in synthesized_df.columns}

# Get shared column names (using train.csv naming convention)
shared_cols = []
for col_lower in train_cols_lower.keys():
    if col_lower in synth_cols_lower:
        train_col = train_cols_lower[col_lower]
        synth_col = synth_cols_lower[col_lower]
        shared_cols.append(train_col)
        
        # Rename synthesized column to match train.csv if different
        if synth_col != train_col:
            synthesized_df = synthesized_df.rename(columns={synth_col: train_col})

print(f"✓ Found {len(shared_cols)} shared columns:")
print(f"  {shared_cols}")

# Select only shared columns from both dataframes
train_df_subset = train_df[shared_cols]
# synthesized_df_subset = synthesized_df[shared_cols]

# Concatenate
# combined_df = pd.concat([train_df_subset, synthesized_df_subset], ignore_index=True)

print(f"\n✓ Combined dataset:")
print(f"  Train data: {len(train_df_subset)} rows")
# print(f"  Synthesized data: {len(synthesized_df_subset)} rows")
print(f"  Total: {len(train_df_subset)} rows")
print(f"  Columns: {list(train_df_subset.columns)}")

✓ Found 8 shared columns:
  ['STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'TASK']

✓ Combined dataset:
  Train data: 2003 rows
  Total: 2003 rows
  Columns: ['STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'TASK']


In [6]:
train_df_subset['ANSWER'] = train_df_subset['ANSWER'].apply(lambda x: f"[[{x}]]")
train_df_subset = train_df_subset.sample(frac=1).reset_index(drop=True)
train_df_subset

/tmp/ipykernel_1064017/563514133.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df_subset['ANSWER'] = train_df_subset['ANSWER'].apply(lambda x: f"[[{x}]]")


,STORY,QUESTION,OPTION-A,OPTION-B,OPTION-C,OPTION-D,ANSWER,TASK
0,Xiao Li and Xiao Wang rest by the swimming poo...,Why does Xiao Wang nudge Xiao Li with his elbow?,Xiao Wang wants Xiao Li to notice the beautifu...,Xiao Wang wants to tell Xiao Li it is time to ...,Xiao Wang wants Xiao Li to notice the obstacle...,Xiao Wang wants Xiao Li to go and chat up Ah Z...,[[D]],Ambiguous Story Task
1,"Zhou Yu wants to go out, but he feels stomacha...",How does Zhou Yu appear when this happens?,Angry,Happy,Sad,Anxious,[[B]],Hidden emotions
2,"The little sister is playing with toys, she se...",What does the little sister really want to say...,The little sister thinks her brother should pl...,The little sister is dissatisfied with her bro...,The little sister wants her brother to let her...,The little sister thinks her brother should tu...,[[C]],Hinting Task Test
3,Zhang Hao's good friend Yang Li shows him a sh...,Why does Zhang Hao say this?,Zhang Hao says this because he truly thinks Ya...,Zhang Hao says this because he maintains a fri...,Zhang Hao is participating in the same competi...,Zhang Hao does not have much insight into lite...,[[B]],Strange Story Task
4,"Xiao Li finds a safe in the laundry room, the ...","After Youyou opens the safe, what does Xiao Li...",Pepper,Vest,Sponge,Dress,[[C]],False Belief Task
...,...,...,...,...,...,...,...,...
1998,Liu Jun and his younger brother have a good re...,Is what the younger brother says true?,Yes,No,NaN,NaN,[[B]],Strange Story Task
1999,Sister Wang Li finds that her younger sister X...,How does Wang Li convince Xiao Yan?,She suggests that she changes her password reg...,She suggests that she discusses this issue wit...,She recommends that she reads some research re...,She reminds her that people on the internet ca...,[[D]],Persuasion Story Task
2000,"At a European medieval art exhibition, An Lei ...",Does Aunt Jane know that Anlei participates in...,Knows,Does not know,NaN,NaN,[[A]],Faux-pas Recognition Test
2001,Xiao Li and Youyou are hanging out in the laun...,Where does Xiaoli look for the sponge after Yo...,Cabinet,Box,Handbag,Storage locker,[[B]],False Belief Task


In [8]:
train_df_subset.to_csv("finetune_data_no_synthesize.csv", index=False)

In [9]:
def create_training_sample(row):
    """
    Convert a single row to Qwen2.5 training format.
    Matches the format from ToM/prompt-noCoT.txt
    """
    # Format the question following prompt-noCoT.txt structure
    user_message = f"""Below is a multiple-choice question with a story and serveral answer options. Based on the content of the story and the given question, please infer the most likely answer and output the answer index.

Note:
(1) Please only output the most likely answer index in the format: [[Answer Index]], for example, if the most likely answer option is 'A. Handbag', then output '[[A]]';
(2) You must choose one of the given answer options 'A, B, C, D' as the most likely answer, regardless of whether the story provides enough information. If you think there is not enough information in the story to choose an answer, please output the most likely answer among "[[A]]", "[[B]]", "[[C]]", or "[[D]]" based on the current story;
(3) Please only output the most likely answer index based on the given information, and do not output any other content.

[Story]
{row['STORY']}

[Question]
{row['QUESTION']}

[Candidate Answers]
A. {row['OPTION-A']} B. {row['OPTION-B']} C. {row['OPTION-C']} D. {row['OPTION-D']}

Output the most likely answer among "[[A]]", "[[B]]", "[[C]]", or "[[D]]", and nothing else. Do not think."""
    
    # The correct answer (already wrapped in [[]] from previous step)
    assistant_message = row['ANSWER']
    
    return {
        "messages": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
    }

# Convert all rows
training_data = [create_training_sample(row) for _, row in train_df_subset.iterrows()]

print(f"✓ Created {len(training_data)} training samples")
print(f"\nExample sample:")
print(json.dumps(training_data[0], indent=2, ensure_ascii=False))

✓ Created 2003 training samples

Example sample:
{
  "messages": [
    {
      "role": "user",
      "content": "Below is a multiple-choice question with a story and serveral answer options. Based on the content of the story and the given question, please infer the most likely answer and output the answer index.\n\nNote:\n(1) Please only output the most likely answer index in the format: [[Answer Index]], for example, if the most likely answer option is 'A. Handbag', then output '[[A]]';\n(2) You must choose one of the given answer options 'A, B, C, D' as the most likely answer, regardless of whether the story provides enough information. If you think there is not enough information in the story to choose an answer, please output the most likely answer among \"[[A]]\", \"[[B]]\", \"[[C]]\", or \"[[D]]\" based on the current story;\n(3) Please only output the most likely answer index based on the given information, and do not output any other content.\n\n[Story]\nXiao Li and Xiao Wang r

In [10]:
from sklearn.model_selection import train_test_split

# Split 90% train, 10% validation
train_data, val_data = train_test_split(training_data, test_size=0.1, random_state=42)

print(f"✓ Split complete:")
print(f"  Train samples: {len(train_data)}")
print(f"  Validation samples: {len(val_data)}")

✓ Split complete:
  Train samples: 1802
  Validation samples: 201


In [11]:
output_dir = Path("../finetune")
output_dir.mkdir(exist_ok=True)

# Save train data
train_path = output_dir / "train_nosyn.jsonl"
with open(train_path, 'w', encoding='utf-8') as f:
    for sample in train_data:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')

# Save validation data
val_path = output_dir / "val_nosyn.jsonl"
with open(val_path, 'w', encoding='utf-8') as f:
    for sample in val_data:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')

print(f"✓ Saved training data to {train_path}")
print(f"✓ Saved validation data to {val_path}")

✓ Saved training data to ../finetune/train_nosyn.jsonl
✓ Saved validation data to ../finetune/val_nosyn.jsonl
